# Task 098: zdipy_doppler — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Zodiacal light Doppler imaging using least-squares inversion

**Paper**: The evolution of surface magnetic fields in young solar-type stars II: the early main sequence (250-650 Myr)
**Repository**: https://github.com/folsomcp/ZDIpy

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 20.66 dB ← 🔧 修复前: 8.89 dB
- **SSIM**: 0.8690
- **Evaluation**: Zeeman-Doppler Imaging stellar surface magnetic mapping

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
zdipy_doppler — Doppler Imaging of Stellar Surfaces
=====================================================
Reconstruct stellar surface brightness from rotationally-broadened spectral
line profiles observed at multiple rotation phases.

Physics:
  - A rapidly rotating star has surface elements Doppler-shifted by
    v_j(phi) = v_eq * sin(i) * cos(lat_j) * sin(lon_j + 2*pi*phi)
  - Each visible element contributes a local line profile weighted by
    its brightness and projected area (limb-darkening via mu = cos theta)
  - Forward: I(v, phi) = Sum_j  B_j * g(v - v_j(phi)) * mu_j * dOmega_j * vis_j
  - Inverse: Tikhonov-regularised least-squares to recover brightness map B_j
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`compute_visibility_and_velocity`, `build_design_matrix`, `compute_cc`, `main`

In [ ]:
# ====================================================================
# 3. Visibility & projected velocity
# ====================================================================
def compute_visibility_and_velocity(lats, lons, phase, inc_rad, v_eq):
    """For a given rotation phase, compute visibility and radial velocity.

    Parameters
    ----------
    phase : float in [0, 1)  rotation phase

    Returns
    -------
    mu    : (N_PIX,)  cos(angle to observer);  >0 means visible
    v_rad : (N_PIX,)  radial velocity of each surface element (km/s)
    """
    # At phase phi the star rotates, equivalent to shifting longitudes
    shifted_lon = lons + 2.0 * np.pi * phase

    # Direction cosine: mu = sin(i)*cos(lat)*cos(shifted_lon) + cos(i)*sin(lat)
    sin_i = np.sin(inc_rad)
    cos_i = np.cos(inc_rad)
    mu = sin_i * np.cos(lats) * np.cos(shifted_lon) + cos_i * np.sin(lats)

    # Radial velocity: projection of rotation onto line of sight
    # v_rad = v_eq * sin(i) * cos(lat) * sin(shifted_lon)
    v_rad = v_eq * sin_i * np.cos(lats) * np.sin(shifted_lon)

    return mu, v_rad

# ====================================================================
# 4. Build design matrix  (Forward model)
# ====================================================================
def build_design_matrix(lats, lons, d_omega, phases, v_axis, v_eq, inc_deg,
                        local_width, limb_eps):
    """Assemble the design matrix A so that  d = A @ B.

    d : (N_PHASES * N_VBINS,)  observed line profile data vector
    B : (N_PIX,)               surface brightness

    A[k, j] = vis_j(phi) * mu_j(phi)^ld * dOmega_j * g(v_k - v_j(phi))
    where g is a Gaussian local line profile, k indexes (phase, v-bin).
    """
    inc_rad = np.radians(inc_deg)
    n_ph = len(phases)
    n_v  = len(v_axis)
    n_pix = len(lats)

    A = np.zeros((n_ph * n_v, n_pix), dtype=np.float64)

    for ip, phi in enumerate(phases):
        mu, v_rad = compute_visibility_and_velocity(lats, lons, phi, inc_rad, v_eq)
        visible = mu > 0.0

        # Limb-darkening weight:  I_ld = 1 - eps*(1 - mu)
        ld_weight = np.where(visible, 1.0 - limb_eps * (1.0 - mu), 0.0)
        weight = ld_weight * d_omega * visible.astype(np.float64)

        for j in range(n_pix):
            if not visible[j]:
                continue
            # Gaussian local line profile centred at v_rad[j]
            profile = np.exp(-0.5 * ((v_axis - v_rad[j]) / local_width) ** 2)
            profile /= (local_width * np.sqrt(2.0 * np.pi))
            A[ip * n_v:(ip + 1) * n_v, j] = weight[j] * profile

    return A

def compute_cc(gt, rec):
    """Pearson correlation coefficient."""
    gt_f = gt.ravel().astype(np.float64)
    rec_f = rec.ravel().astype(np.float64)
    gt_z = gt_f - gt_f.mean()
    rec_z = rec_f - rec_f.mean()
    denom = np.linalg.norm(gt_z) * np.linalg.norm(rec_z)
    if denom < 1e-30:
        return 0.0
    return float(np.dot(gt_z, rec_z) / denom)

# ====================================================================
# 9. Main pipeline
# ====================================================================
def main():
    t0 = time.time()
    sep = "=" * 70
    print(sep)
    print("Doppler Imaging — Stellar Surface Brightness Recovery")
    print(sep)

    # ── 1  surface grid ──
    print("\n[1/7] Creating surface grid ...")
    lats, lons, d_omega = create_surface_grid(N_LAT, N_LON)
    print(f"  {N_LAT} lat x {N_LON} lon = {N_PIX} surface elements")

    # ── 2  ground truth ──
    print("\n[2/7] Ground-truth brightness map (3 dark spots) ...")
    B_gt = create_ground_truth(lats, lons)
    n_spot = int(np.sum(B_gt < 0.9))
    print(f"  Photosphere=1.0,  {n_spot} spot pixels")
    print(f"  B range = [{B_gt.min():.2f}, {B_gt.max():.2f}]")

    # ── 3  observation setup ──
    print("\n[3/7] Observation setup ...")
    phases = np.linspace(0, 1.0, N_PHASES, endpoint=False)
    v_axis = np.linspace(-V_MAX, V_MAX, N_VBINS)
    print(f"  {N_PHASES} phases,  {N_VBINS} velocity bins  +/-{V_MAX} km/s")
    print(f"  v_eq={V_EQ} km/s,  inc={INCLINATION} deg,  eps_LD={LIMB_DARK}")

    # ── 4  design matrix ──
    print("\n[4/7] Building design matrix ...")
    A = build_design_matrix(lats, lons, d_omega, phases, v_axis,
                            V_EQ, INCLINATION, LOCAL_WIDTH, LIMB_DARK)
    print(f"  A shape = {A.shape}  (data={N_PHASES * N_VBINS}, pixels={N_PIX})")
    print(f"  A nonzero fraction = {np.count_nonzero(A) / A.size:.4f}")

    # ── 5  forward + noise ──
    print("\n[5/7] Forward solve + noise ...")
    d_clean = forward_solve(A, B_gt)
    noise_power = np.linalg.norm(d_clean) / (10 ** (SNR_DB / 20))
    rng = np.random.default_rng(42)
    noise = rng.standard_normal(d_clean.shape)
    noise *= noise_power / np.linalg.norm(noise)
    d_noisy = d_clean + noise
    print(f"  |d_clean| = {np.linalg.norm(d_clean):.6e}")
    print(f"  |noise|   = {np.linalg.norm(noise):.6e}")
    print(f"  SNR       = {SNR_DB} dB")

    # ── 6  inverse solve ──
    print("\n[6/7] Tikhonov inversion ...")
    B_rec, lam_used = tikhonov_solve(A, d_noisy, LAMBDA_REG)
    # Clip to physical range [0, 1]
    B_rec = np.clip(B_rec, 0.0, 1.0)
    print(f"  Lambda = {lam_used:.6e}")
    print(f"  B_rec range = [{B_rec.min():.4f}, {B_rec.max():.4f}]")

    # ── 7  metrics ──
    print("\n[7/7] Metrics & visualisation ...")
    psnr_val = compute_psnr(B_gt, B_rec)
    ssim_val = compute_ssim(B_gt, B_rec)
    cc_val   = compute_cc(B_gt, B_rec)
    rmse_val = compute_rmse(B_gt, B_rec)
    metrics = {
        "PSNR": psnr_val,
        "SSIM": ssim_val,
        "CC":   cc_val,
        "RMSE": rmse_val,
    }
    print(f"  PSNR = {psnr_val:.2f} dB")
    print(f"  SSIM = {ssim_val:.4f}")
    print(f"  CC   = {cc_val:.4f}")
    print(f"  RMSE = {rmse_val:.6f}")

    # ── save arrays ──
    np.save(os.path.join(RESULTS_DIR, "gt_output.npy"), B_gt.reshape(N_LAT, N_LON))
    np.save(os.path.join(RESULTS_DIR, "recon_output.npy"), B_rec.reshape(N_LAT, N_LON))
    np.save(os.path.join(RESULTS_DIR, "data_noisy.npy"), d_noisy.reshape(N_PHASES, N_VBINS))
    np.save(os.path.join(RESULTS_DIR, "design_matrix.npy"), A)

    # website assets
    np.save(os.path.join(ASSETS_DIR, "gt_output.npy"), B_gt.reshape(N_LAT, N_LON))
    np.save(os.path.join(ASSETS_DIR, "recon_output.npy"), B_rec.reshape(N_LAT, N_LON))

    # metrics JSON
    with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    with open(os.path.join(ASSETS_DIR, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    # plots
    vis_paths = [
        os.path.join(RESULTS_DIR, "vis_result.png"),
        os.path.join(ASSETS_DIR, "vis_result.png"),
        os.path.join(WORKING_DIR, "vis_result.png"),
    ]
    plot_results(B_gt, B_rec, N_LAT, N_LON, metrics, vis_paths)

    elapsed = time.time() - t0
    print(f"\n{sep}")
    print(f"DONE ({elapsed:.1f}s)  PSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}  CC={cc_val:.4f}")
    print(sep)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `create_surface_grid`, `create_ground_truth`, `forward_solve`

In [ ]:
# ====================================================================
# 1. Surface grid (latitude-longitude)
# ====================================================================
def create_surface_grid(n_lat, n_lon):
    """Equal-area-ish lat/lon grid on the stellar surface.

    Returns
    -------
    lats    : (N_PIX,)  latitudes in radians [-pi/2, pi/2]
    lons    : (N_PIX,)  longitudes in radians [0, 2*pi)
    d_omega : (N_PIX,)  solid-angle element for each pixel
    """
    lat_edges = np.linspace(-np.pi / 2, np.pi / 2, n_lat + 1)
    lon_edges = np.linspace(0, 2 * np.pi, n_lon + 1)

    lats_1d = 0.5 * (lat_edges[:-1] + lat_edges[1:])
    lons_1d = 0.5 * (lon_edges[:-1] + lon_edges[1:])

    LONS, LATS = np.meshgrid(lons_1d, lats_1d)  # (n_lat, n_lon)
    lats = LATS.ravel()
    lons = LONS.ravel()

    # solid-angle element  dOmega = cos(lat) * d_lat * d_lon
    d_lat = lat_edges[1] - lat_edges[0]
    d_lon = lon_edges[1] - lon_edges[0]
    d_omega = np.abs(np.cos(lats)) * d_lat * d_lon

    return lats, lons, d_omega

# ====================================================================
# 2. Ground-truth brightness map (star with dark spots)
# ====================================================================
def create_ground_truth(lats, lons):
    """Create a brightness map with 3 dark spots on a bright photosphere.

    Returns
    -------
    B_gt : (N_PIX,) surface brightness (1 = photosphere, <1 = spot)
    """
    B_gt = np.ones(len(lats), dtype=np.float64)

    # Spot 1: large cool spot near equator
    lat_c1, lon_c1 = np.radians(10), np.radians(90)
    r1, depth1 = np.radians(20), 0.3
    ang1 = np.arccos(np.clip(
        np.sin(lats) * np.sin(lat_c1) +
        np.cos(lats) * np.cos(lat_c1) * np.cos(lons - lon_c1), -1, 1))
    B_gt[ang1 < r1] = depth1

    # Spot 2: mid-latitude spot
    lat_c2, lon_c2 = np.radians(40), np.radians(220)
    r2, depth2 = np.radians(15), 0.4
    ang2 = np.arccos(np.clip(
        np.sin(lats) * np.sin(lat_c2) +
        np.cos(lats) * np.cos(lat_c2) * np.cos(lons - lon_c2), -1, 1))
    B_gt[ang2 < r2] = depth2

    # Spot 3: small polar spot
    lat_c3, lon_c3 = np.radians(65), np.radians(350)
    r3, depth3 = np.radians(12), 0.5
    ang3 = np.arccos(np.clip(
        np.sin(lats) * np.sin(lat_c3) +
        np.cos(lats) * np.cos(lat_c3) * np.cos(lons - lon_c3), -1, 1))
    B_gt[ang3 < r3] = depth3

    return B_gt

# ====================================================================
# 5. Forward solve
# ====================================================================
def forward_solve(A, B):
    """Compute observed line profiles  d = A @ B."""
    return A @ B

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `tikhonov_solve`

In [ ]:
# ====================================================================
# 6. Tikhonov inverse solve
# ====================================================================
def tikhonov_solve(A, d, lam):
    """Solve  min ||A B - d||^2 + lambda ||B - 1||^2   (default brightness = 1).

    Using the substitution  B' = B - 1:
        min ||A B' - (d - A*1)||^2 + lambda*||B'||^2

    Returns
    -------
    B_rec    : (N_PIX,)  reconstructed brightness
    lam_used : float
    """
    n_pix = A.shape[1]
    ones_vec = np.ones(n_pix)
    d_shifted = d - A @ ones_vec  # shifted data

    AtA = A.T @ A
    Atd = A.T @ d_shifted

    if lam == "auto":
        # Heuristic: lambda = trace(AtA) / n_pix * factor
        lam_used = np.trace(AtA) / n_pix * 0.3
    else:
        lam_used = float(lam)

    B_prime = np.linalg.solve(AtA + lam_used * np.eye(n_pix), Atd)
    B_rec = B_prime + ones_vec

    return B_rec, lam_used

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_psnr`, `compute_ssim`, `compute_rmse`, `plot_results`

In [ ]:
# ====================================================================
# 7. Metrics
# ====================================================================
def compute_psnr(gt, rec):
    """Peak signal-to-noise ratio."""
    data_range = gt.max() - gt.min()
    if data_range < 1e-12:
        return 0.0
    mse = np.mean((gt - rec) ** 2)
    if mse < 1e-30:
        return 100.0
    return 10.0 * np.log10(data_range ** 2 / mse)

def compute_ssim(gt, rec, C1=None, C2=None):
    """Simple SSIM between 1-D or 2-D arrays."""
    gt_f = gt.ravel().astype(np.float64)
    rec_f = rec.ravel().astype(np.float64)
    drange = gt_f.max() - gt_f.min()
    if C1 is None:
        C1 = (0.01 * drange) ** 2
    if C2 is None:
        C2 = (0.03 * drange) ** 2
    mu_x = gt_f.mean()
    mu_y = rec_f.mean()
    sig_x = gt_f.std()
    sig_y = rec_f.std()
    sig_xy = np.mean((gt_f - mu_x) * (rec_f - mu_y))
    num = (2 * mu_x * mu_y + C1) * (2 * sig_xy + C2)
    den = (mu_x ** 2 + mu_y ** 2 + C1) * (sig_x ** 2 + sig_y ** 2 + C2)
    return float(num / den)

def compute_rmse(gt, rec):
    """Root mean square error."""
    return float(np.sqrt(np.mean((gt - rec) ** 2)))

# ====================================================================
# 8. Visualisation
# ====================================================================
def plot_results(B_gt, B_rec, n_lat, n_lon, metrics, save_paths):
    """Plot ground truth vs reconstruction as Mollweide maps."""
    B_gt_2d  = B_gt.reshape(n_lat, n_lon)
    B_rec_2d = B_rec.reshape(n_lat, n_lon)

    lat_edges = np.linspace(-np.pi / 2, np.pi / 2, n_lat + 1)
    lon_edges = np.linspace(0, 2 * np.pi, n_lon + 1)
    lats_1d = 0.5 * (lat_edges[:-1] + lat_edges[1:])
    lons_1d = 0.5 * (lon_edges[:-1] + lon_edges[1:]) - np.pi  # shift to [-pi, pi]

    LON, LAT = np.meshgrid(lons_1d, lats_1d)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5),
                             subplot_kw={"projection": "mollweide"})

    # GT
    im0 = axes[0].pcolormesh(LON, LAT, B_gt_2d, cmap="inferno",
                             vmin=0, vmax=1, shading="auto")
    axes[0].set_title("Ground Truth", fontsize=12, fontweight="bold")
    axes[0].grid(True, alpha=0.3)
    plt.colorbar(im0, ax=axes[0], orientation="horizontal", pad=0.05, shrink=0.8)

    # Reconstruction
    im1 = axes[1].pcolormesh(LON, LAT, np.clip(B_rec_2d, 0, 1), cmap="inferno",
                             vmin=0, vmax=1, shading="auto")
    axes[1].set_title("Reconstruction", fontsize=12, fontweight="bold")
    axes[1].grid(True, alpha=0.3)
    plt.colorbar(im1, ax=axes[1], orientation="horizontal", pad=0.05, shrink=0.8)

    # Residual
    residual = np.abs(B_gt_2d - B_rec_2d)
    im2 = axes[2].pcolormesh(LON, LAT, residual, cmap="hot",
                             vmin=0, vmax=0.5, shading="auto")
    axes[2].set_title("|Residual|", fontsize=12, fontweight="bold")
    axes[2].grid(True, alpha=0.3)
    plt.colorbar(im2, ax=axes[2], orientation="horizontal", pad=0.05, shrink=0.8)

    fig.suptitle(
        f"Doppler Imaging — Stellar Surface Brightness Recovery\n"
        f"PSNR={metrics['PSNR']:.2f} dB   SSIM={metrics['SSIM']:.4f}   "
        f"CC={metrics['CC']:.4f}",
        fontsize=13, fontweight="bold", y=1.04,
    )
    plt.tight_layout()
    for p in save_paths:
        os.makedirs(os.path.dirname(p), exist_ok=True)
        fig.savefig(p, dpi=150, bbox_inches="tight")
        print(f"  Saved -> {p}")
    plt.close(fig)

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
t0 = time.time()
sep = "=" * 70
print(sep)
print("Doppler Imaging — Stellar Surface Brightness Recovery")
print(sep)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ── 1  surface grid ──
print("\n[1/7] Creating surface grid ...")
lats, lons, d_omega = create_surface_grid(N_LAT, N_LON)
print(f"  {N_LAT} lat x {N_LON} lon = {N_PIX} surface elements")

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# ── 2  ground truth ──
print("\n[2/7] Ground-truth brightness map (3 dark spots) ...")
B_gt = create_ground_truth(lats, lons)
n_spot = int(np.sum(B_gt < 0.9))
print(f"  Photosphere=1.0,  {n_spot} spot pixels")
print(f"  B range = [{B_gt.min():.2f}, {B_gt.max():.2f}]")

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# ── 3  observation setup ──
print("\n[3/7] Observation setup ...")
phases = np.linspace(0, 1.0, N_PHASES, endpoint=False)
v_axis = np.linspace(-V_MAX, V_MAX, N_VBINS)
print(f"  {N_PHASES} phases,  {N_VBINS} velocity bins  +/-{V_MAX} km/s")
print(f"  v_eq={V_EQ} km/s,  inc={INCLINATION} deg,  eps_LD={LIMB_DARK}")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ── 4  design matrix ──
print("\n[4/7] Building design matrix ...")
A = build_design_matrix(lats, lons, d_omega, phases, v_axis,
                        V_EQ, INCLINATION, LOCAL_WIDTH, LIMB_DARK)
print(f"  A shape = {A.shape}  (data={N_PHASES * N_VBINS}, pixels={N_PIX})")
print(f"  A nonzero fraction = {np.count_nonzero(A) / A.size:.4f}")

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# ── 5  forward + noise ──
print("\n[5/7] Forward solve + noise ...")
d_clean = forward_solve(A, B_gt)
noise_power = np.linalg.norm(d_clean) / (10 ** (SNR_DB / 20))
rng = np.random.default_rng(42)
noise = rng.standard_normal(d_clean.shape)
noise *= noise_power / np.linalg.norm(noise)
d_noisy = d_clean + noise
print(f"  |d_clean| = {np.linalg.norm(d_clean):.6e}")
print(f"  |noise|   = {np.linalg.norm(noise):.6e}")
print(f"  SNR       = {SNR_DB} dB")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ── 6  inverse solve ──
print("\n[6/7] Tikhonov inversion ...")
B_rec, lam_used = tikhonov_solve(A, d_noisy, LAMBDA_REG)
# Clip to physical range [0, 1]
B_rec = np.clip(B_rec, 0.0, 1.0)
print(f"  Lambda = {lam_used:.6e}")
print(f"  B_rec range = [{B_rec.min():.4f}, {B_rec.max():.4f}]")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── 7  metrics ──
print("\n[7/7] Metrics & visualisation ...")
psnr_val = compute_psnr(B_gt, B_rec)
ssim_val = compute_ssim(B_gt, B_rec)
cc_val   = compute_cc(B_gt, B_rec)
rmse_val = compute_rmse(B_gt, B_rec)
metrics = {
    "PSNR": psnr_val,
    "SSIM": ssim_val,
    "CC":   cc_val,
    "RMSE": rmse_val,
}
print(f"  PSNR = {psnr_val:.2f} dB")
print(f"  SSIM = {ssim_val:.4f}")
print(f"  CC   = {cc_val:.4f}")
print(f"  RMSE = {rmse_val:.6f}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ── save arrays ──
np.save(os.path.join(RESULTS_DIR, "gt_output.npy"), B_gt.reshape(N_LAT, N_LON))
np.save(os.path.join(RESULTS_DIR, "recon_output.npy"), B_rec.reshape(N_LAT, N_LON))
np.save(os.path.join(RESULTS_DIR, "data_noisy.npy"), d_noisy.reshape(N_PHASES, N_VBINS))
np.save(os.path.join(RESULTS_DIR, "design_matrix.npy"), A)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# website assets
np.save(os.path.join(ASSETS_DIR, "gt_output.npy"), B_gt.reshape(N_LAT, N_LON))
np.save(os.path.join(ASSETS_DIR, "recon_output.npy"), B_rec.reshape(N_LAT, N_LON))

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# metrics JSON
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
with open(os.path.join(ASSETS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# plots
vis_paths = [
    os.path.join(RESULTS_DIR, "vis_result.png"),
    os.path.join(ASSETS_DIR, "vis_result.png"),
    os.path.join(WORKING_DIR, "vis_result.png"),
]
plot_results(B_gt, B_rec, N_LAT, N_LON, metrics, vis_paths)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
elapsed = time.time() - t0
print(f"\n{sep}")
print(f"DONE ({elapsed:.1f}s)  PSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}  CC={cc_val:.4f}")
print(sep)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **zdipy_doppler**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=20.66 dB ← 🔧 修复前: 8.89 dB, SSIM=0.8690)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: The evolution of surface magnetic fields in young solar-type stars II: the early main sequence (250-650 Myr)
- Repository: https://github.com/folsomcp/ZDIpy
